### Normalize and correct sgRNA Log Fold Change data
#### adapted from Fortin et al, 2019
#### by Stefanus Bernard

In [ ]:
import pandas as pd
from normalize_lfc_utils import *

In [ ]:
lfc_data = pd.read_csv("../../data/sgrna_lfc_data/jacquere_data/jacquereA375lfc.csv", sep =',')
lfc_data = lfc_data.rename(columns={lfc_data.columns[1]: 'spacer', lfc_data.columns[6]: 'gene'})
# Select only the 'spacer' and 'sgRNA_lfc' columns
lfc_data = lfc_data[['spacer', 'sgRNA_lfc']]
print(lfc_data.shape)

lfc_data.head(25)

In [ ]:
# import library data
library_data = pd.read_csv("../../data/library_data/restricted_library/Jacquere_PerGuideAnnotations_Quota4.tsv", sep ="\t", header = None)
library_data.columns = ['sgRNA', 'spacer', 'gene']
display(library_data)

In [ ]:
lfc = pd.merge(lfc_data, library_data, how="left", on="spacer").set_index(['sgRNA', 'spacer', 'gene']).sort_values(by="gene")

# Remove duplicate rows
lfc = lfc[~lfc.index.duplicated(keep='first')]
lfc = lfc.sort_index(level='sgRNA')

lfc = lfc.dropna()
print(f"DataFrame shape after dropping NaN values: {lfc.shape}")
lfc

In [ ]:
# Center LFCs using non-essential guides
non_essential_mask = ~lfc.index.get_level_values(2).isin(['NO_SITE'])
median_non_essential = lfc[non_essential_mask].median(axis=0)

median_non_essential

In [ ]:
# Subtract median of non-essential guides for each sample (normalized sgrna LFC)
sgrna_lfc_normalized = lfc - median_non_essential
sgrna_lfc_normalized

In [ ]:
# scaled sgRNA log fold change by centering around the median LFC of guides targeting essential genes
lfc_norm_scaled, list_common_essential = scale_essential(sgrna_lfc_normalized, '../../data/sgrna_lfc_data/constitutive_core_essential_hart_2014.csv')
lfc_norm_scaled

In [ ]:
# check whether the normalization works (the median of lfc data should be around 0)
sanity_check_scale_essential(sgrna_lfc_normalized, lfc_norm_scaled, list_common_essential)

In [ ]:
lfc_norm_scaled_avg = mean_row(lfc_norm_scaled)
lfc_norm_scaled_avg = lfc_norm_scaled_avg.reset_index()
lfc_norm_scaled_avg

In [ ]:
lfc_norm_scaled_avg.to_csv("../../data/sgrna_lfc_data/output_normalized/jacquere_normalized_lfc.csv", index=False)